# RIDE IT: Drivers Engagement Analysis
### End-to-End Operational Analytics, Driver Lifecycle & Performance Study
**Tools:** Python (Pandas, NumPy, Matplotlib, Seaborn), SQL, Power BI, PowerPoint

---

## Phase 2: Python Setup & Libraries
Importing core data manipulation and visualization packages.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.sans-serif'] = 'Arial'
print('Libraries successfully imported.')

## Phase 3 & 4: Ingestion & Data Quality Cleaning
Loading datasets, imputing missing values, and validating funnel constraints.

In [ ]:
drivers = pd.read_csv('../data/Rideit_drivers.csv')
activity = pd.read_csv('../data/Rideit_drivers_activity.csv')

print(f'Drivers dimensions: {drivers.shape}')
print(f'Activity dimensions: {activity.shape}')

drivers['date_registration'] = pd.to_datetime(drivers['date_registration'])
activity['active_date'] = pd.to_datetime(activity['active_date'])

drivers['driver_rating'] = drivers['driver_rating'].fillna(drivers['driver_rating'].median())
drivers['gold_level_count'] = drivers['gold_level_count'].fillna(0)
drivers['receive_marketing'] = drivers['receive_marketing'].fillna(False).astype(bool)

activity['offers_adjusted'] = np.maximum(activity['offers'], activity['bookings'])
activity['rides_adjusted'] = np.minimum(activity['rides'], activity['bookings'])
activity['total_cancellations'] = activity['bookings_cancelled_by_passenger'] + activity['bookings_cancelled_by_driver']

display(drivers.head(3))
display(activity.head(3))

## Phase 5 & 6: Feature Engineering & Merging
Calculating Acceptance Rate, Completion Rate, Cancellation Rate, Driver Tenure, and Engagement Score.

In [ ]:
merged = pd.read_csv('../output/rideit_merged_cleaned.csv')
merged['active_date'] = pd.to_datetime(merged['active_date'])
merged['date_registration'] = pd.to_datetime(merged['date_registration'])

print(f'Merged Master Dataset Shape: {merged.shape}')
print('\nEngagement Tier Breakdown (%):')
print(merged['engagement_tier'].value_counts(normalize=True) * 100)
display(merged[['acceptance_rate', 'completion_rate', 'cancellation_rate', 'engagement_score']].describe())

## Phase 7: Temporal Analysis (Monthly Trends)
Tracking driver engagement and ride trends throughout 2020.

In [ ]:
monthly_kpi = pd.read_csv('../output/monthly_kpi_summary.csv')
display(monthly_kpi)

fig, ax1 = plt.subplots(figsize=(11, 5))
ax2 = ax1.twinx()
sns.lineplot(data=monthly_kpi, x='year_month', y='active_drivers', ax=ax1, color='#1f77b4', marker='o', label='Active Drivers')
sns.lineplot(data=monthly_kpi, x='year_month', y='total_rides', ax=ax2, color='#2ca02c', marker='s', label='Total Completed Rides')
ax1.set_title('RIDE IT - Monthly Active Drivers vs Completed Rides (2020)', fontsize=13, fontweight='bold')
ax1.set_xlabel('Month')
ax1.set_ylabel('Active Drivers', color='#1f77b4')
ax2.set_ylabel('Total Completed Rides', color='#2ca02c')
ax1.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## Phase 8: Segment Factors (TAXI vs PHV, Marketing, Rating, Country)
Analyzing how vehicle category and driver attributes influence engagement.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
sns.barplot(data=merged, x='service_type', y='engagement_score', palette='Blues_d', ax=ax1)
ax1.set_title('Engagement Score: TAXI vs PHV', fontweight='bold')
ax1.set_ylim(0, 100)

sns.barplot(data=merged, x='receive_marketing', y='engagement_score', palette='Purples_d', ax=ax2)
ax2.set_title('Engagement Score by Marketing Opt-In', fontweight='bold')
ax2.set_ylim(0, 100)
plt.tight_layout()
plt.show()

## Phase 9 - 11: Unique Analyses (Driver Journey, Cancellations & Gold Tier)
Lifecycle trajectory, cancellation impact, and loyalty performance.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
tenure_order = ['New (0-1 Month)', 'Early-Stage (1-3 Months)', 'Established (3-12 Months)', 'Veteran (1+ Years)']
sns.barplot(data=merged, x='tenure_group', y='engagement_score', order=tenure_order, palette='Blues_d', ax=ax1)
ax1.set_title('Engagement Across Driver Lifecycle Journey', fontweight='bold')
ax1.tick_params(axis='x', rotation=20)
ax1.set_ylim(0, 100)

cancel_summary = pd.DataFrame({
    'Type': ['Passenger Cancellations', 'Driver Cancellations'],
    'Count': [merged['bookings_cancelled_by_passenger'].sum(), merged['bookings_cancelled_by_driver'].sum()]
})
sns.barplot(data=cancel_summary, x='Type', y='Count', palette=['#3498db', '#e74c3c'], ax=ax2)
ax2.set_title('Cancellation Friction: Passenger vs Driver', fontweight='bold')
plt.tight_layout()
plt.show()